In [ ]:
# ---------------- Imports ----------------
import os
import pandas as pd
import json
import re
from datetime import datetime

import yaml



In [ ]:
# ---------------- Args ----------------
results_file_name = "Batch-5365353-batch-results"




In [ ]:
# ---------------- Config ----------------
with open("../../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj_store"]
models_folderpath = config["paths"]["models"]

timestamp = datetime.now().strftime("%Y%m%dt%H%M%S")

human_eval_dir = os.path.join(proj_store, "evaluation", "human-evaluation")

results_dir = os.path.join(human_eval_dir, "raw-results")
results_file = os.path.join(results_dir, f"{results_file_name}.csv")

formatted_results_dir = os.path.join(human_eval_dir, "formatted-results")
os.makedirs(formatted_results_dir, exist_ok=True)
formatted_results_file = os.path.join(formatted_results_dir, f"{results_file_name}-formatted.csv")


analysis_results_dir = os.path.join(human_eval_dir, "analysis")
os.makedirs(analysis_results_dir, exist_ok=True)



## Formatting

In [ ]:
results_table = pd.read_csv(results_file)

display(results_table.columns)
display(results_table.shape)
display(results_table.head(2))


In [ ]:
results_table = results_table[
    [
        "HITId",
        "AssignmentId",
        "Input.block_id",
        "Input.domain",
        "Input.context_turns",
        "Input.response_1",
        "Input.response_1_label",
        "Input.response_2",
        "Input.response_2_label",
        "Input.response_3",
        "Input.response_3_label",
        "Input.response_4",
        "Input.response_4_label",
        "Answer.taskAnswers",

        ]
    ]


display(results_table.head(2))

In [ ]:
# mapping from metric suffix -> Input column prefix
suffix_to_input = {
    1: "Input.response_1",
    2: "Input.response_2",
    3: "Input.response_3",
    4: "Input.response_4",
}

def parse_task_answers(task_str):
    if pd.isna(task_str):
        return {}
    data = json.loads(task_str)
    if isinstance(data, list) and data:
        data = data[0]
    # for each metric dict, pick the key where value is True and cast to int
    out = {}
    for metric, choices in data.items():
        chosen = next((k for k, v in choices.items() if v), None)
        out[metric] = int(chosen) if chosen is not None else None
    return out

# 1) expand the metrics to columns like Conformity1, Control2, ...
expanded = results_table["Answer.taskAnswers"].apply(parse_task_answers).apply(pd.Series)

# 2) rename using the suffix->Input mapping
def rename_with_input(col):
    m = re.search(r'(\d+)$', col)
    if not m:
        return col
    idx = int(m.group(1))
    prefix = suffix_to_input.get(idx)
    return f"{prefix}.{col}" if prefix else col

expanded.columns = [rename_with_input(c) for c in expanded.columns]

# 3) combine with the original table
new_df = pd.concat([results_table.drop(columns=["Answer.taskAnswers"]), expanded], axis=1)

display(new_df.head(2))




In [ ]:
# Remove "Input" from all column names
new_df.columns = new_df.columns.str.replace("^Input\.?", "", regex=True)



new_df = new_df.rename(columns={
    'HITId': 'HITId',
    'AssignmentId': 'AssignmentId',
    'block_id': 'block_id',
    'domain': 'domain',
    'context_turns': 'context_turns',
    'response_1': 'response_1',
    'response_2': 'response_2',
    'response_3': 'response_3',
    'response_4': 'response_4',
    'response_1.Conformity1': 'response.Conformity_1',
    'response_2.Conformity2': 'response.Conformity_2',
    'response_3.Conformity3': 'response.Conformity_3',
    'response_4.Conformity4': 'response.Conformity_4',
    'response_1.Control1': 'response.Control_1',
    'response_2.Control2': 'response.Control_2',
    'response_3.Control3': 'response.Control_3',
    'response_4.Control4': 'response.Control_4',
    'response_1.GoalRel1': 'response.GoalRel_1',
    'response_2.GoalRel2': 'response.GoalRel_2',
    'response_3.GoalRel3': 'response.GoalRel_3',
    'response_4.GoalRel4': 'response.GoalRel_4',
    'response_1.Probing1': 'response.Probing_1',
    'response_2.Probing2': 'response.Probing_2',
    'response_3.Probing3': 'response.Probing_3',
    'response_4.Probing4': 'response.Probing_4',
    'response_1.Progression1': 'response.Progression_1',
    'response_2.Progression2': 'response.Progression_2',
    'response_3.Progression3': 'response.Progression_3',
    'response_4.Progression4': 'response.Progression_4',
    'response_1_label': 'response.label_1',
    'response_2_label': 'response.label_2',
    'response_3_label': 'response.label_3',
    'response_4_label': 'response.label_4',
})


display(new_df.columns)
display(new_df.head(2))




In [ ]:
new_df_arrange = new_df[
    [
        'AssignmentId',
        'HITId',
        'block_id',
        'domain',
        'context_turns',
        'response_1',
        'response.label_1',
        'response.Conformity_1',
        'response.Control_1',
        'response.GoalRel_1',
        'response.Probing_1',
        'response.Progression_1',
        'response_2',
        'response.label_2',
        'response.Conformity_2',
        'response.Control_2',
        'response.GoalRel_2',
        'response.Probing_2',
        'response.Progression_2',
        'response_3',
        'response.label_3',
        'response.Conformity_3',
        'response.Control_3',
        'response.GoalRel_3',
        'response.Probing_3',
        'response.Progression_3',
        'response_4',
        'response.label_4',
        'response.Conformity_4',
        'response.Control_4',
        'response.GoalRel_4',
        'response.Probing_4',
        'response.Progression_4',
        ]
    ]


display(new_df_arrange.head(5))



In [ ]:
# Reshape from wide to long
long_df = pd.wide_to_long(
    new_df_arrange,
    stubnames=[
        "response", "response.label",
        "response.Conformity", "response.Control",
        "response.GoalRel", "response.Probing",
        "response.Progression"
    ],
    i=["AssignmentId", "HITId", "block_id", "domain", "context_turns"],
    j="response_number",
    sep="_",
    suffix="\\d+"
).reset_index()

# Drop the response_number if you don’t care which (1,2,3) it was
long_df = long_df.drop(columns=["response_number"])

display(long_df.head())




In [ ]:
# Remove "response." (with the dot) from all column names
long_df.columns = long_df.columns.str.replace("response\.", "", regex=True)

display(long_df.head(2))


In [ ]:
# Sort by block_id first, then by response.label
sorted_df = long_df.sort_values(
    by=["block_id", "label"],
    ascending=[True, True]  # change to False if you want descending
).reset_index(drop=True)



display(sorted_df.columns)
display(sorted_df.head())



In [ ]:
sorted_df = sorted_df.rename(columns={
    'AssignmentId': 'assignment_id',
    'HITId': 'hit_id',
    'block_id': 'block_id',
    'domain': 'domain',
    'context_turns': 'context_turns',
    'response': 'response',
    'label': 'label',
    'Conformity': 'conformity',
    'Control': 'control',
    'GoalRel': 'goal_rel',
    'Probing': 'probing',
    'Progression': 'progression',
})

display(sorted_df.head())


In [ ]:
sorted_df.to_csv(formatted_results_file, header=True)



## Analysis

In [ ]:
# Clean up the label column
sorted_df["label"] = (
    sorted_df["label"]
    .str.replace("_", "-", regex=False)  # swap underscores for spaces
    .str.title()                         # capitalize each word
    .str.replace("Sft", "SFT", regex=False)  # swap underscores for spaces
    .str.replace("Orl", "ORL", regex=False)  # swap underscores for spaces
)

sorted_df = sorted_df.rename(columns={
    'domain': 'Domain',
    'label': 'Label',
    'conformity': 'Conformity',
    'control': 'Conversational Control',
    'goal_rel': 'Outcome Relevance',
    'probing': 'Probing Effectiveness',
    'progression': 'Progression',
})

display(sorted_df.head())


In [ ]:

# Get the list of unique labels
labels = sorted_df["Label"].unique()

tables = {}
for lbl in labels:
    subset = sorted_df[sorted_df["Label"] == lbl]

    # Group by domain, compute mean, transpose, and round
    summary = (
        subset.groupby("Domain")[[
            "Conformity", 
            "Progression", 
            "Outcome Relevance", 
            "Probing Effectiveness",  
            "Conversational Control"
        ]]
        .mean()
        .round(4)
        #.T
    )

    # Remove the rogue "Domain" header
    summary.columns.name = None

    # Turn the index into a column called "Metric"
    summary = summary.reset_index().rename(columns={"index": "Metric"})

    tables[lbl] = summary
    
    # Export directly to CSV (one file per label)
    clean_lbl = lbl.lower().replace("-", "_")
    filename = f"{results_file_name}-{clean_lbl}.csv"
    summary.to_csv(os.path.join(analysis_results_dir, filename), index=False)
    print(f"Saved {filename}")


# -------------------------------
# Create the "all" table (ignore labels)
# -------------------------------
all_summary = (
    sorted_df.groupby("Domain")[[
        "Progression", 
        "Conversational Control",
        "Outcome Relevance", 
        "Probing Effectiveness",  
        "Conformity", 
    ]]
    .mean()
    .round(4)
    #.T
)

all_summary.columns.name = None
all_summary = all_summary.reset_index().rename(columns={"index": "Metric"})

tables["all"] = all_summary

# Export "all" table
all_filename = f"{results_file_name}-all.csv"
all_summary.to_csv(os.path.join(analysis_results_dir, all_filename), index=False)
print(f"Saved {all_filename}")


# Example: display the "all" table
display("Table for all (no label distinction):")
display(tables["all"])


In [ ]:
# Define the desired order
label_order = ["Real", "Prompted", "SFT", "ORL"]

# Convert Label column to categorical with the custom order
sorted_df["Label"] = pd.Categorical(sorted_df["Label"], categories=label_order, ordered=True)

# Group by label and calculate mean of each metric
summary = (
    sorted_df.groupby(
        "Label", observed=True
    )[["Progression", "Conversational Control", "Outcome Relevance", "Probing Effectiveness", "Conformity"]]
      .mean()
      .round(4)
      .reset_index()
)

filename = f"{results_file_name}-per-label.csv"
summary.to_csv(os.path.join(analysis_results_dir, filename), index=False)
    
display(summary)


## inter-annotator agreement

In [ ]:
import numpy as np
import krippendorff

# %%
# ---------------- Krippendorff's Alpha ----------------

alpha_results = {}

metrics = [
    "Conformity",
    "Conversational Control",
    "Outcome Relevance",
    "Probing Effectiveness",
    "Progression"
]

item_cols = ["block_id", "response", "Label"]

for metric in metrics:
    # Pivot to item × annotator
    pivot = sorted_df.pivot_table(
        index=item_cols,
        columns="assignment_id",
        values=metric,
        observed=True
    )

    # Convert to (raters × items)
    data = pivot.to_numpy().T

    # Compute alpha (ordinal)
    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement="ordinal"
    )

    alpha_results[metric] = alpha

# Convert to DataFrame
alpha_df = (
    pd.DataFrame.from_dict(alpha_results, orient="index", columns=["Krippendorff_alpha"])
      .reset_index()
      .rename(columns={"index": "Metric"})
      .round(4)
)

display(alpha_df)


In [ ]:
# %%
# ---------------- Krippendorff's Alpha per Label (no pivot) ----------------

import numpy as np
import krippendorff

metrics = [
    "Conformity",
    "Conversational Control",
    "Outcome Relevance",
    "Probing Effectiveness",
    "Progression"
]

alpha_rows = []

for label in sorted_df["Label"].cat.categories:
    df_label = sorted_df[sorted_df["Label"] == label]

    for metric in metrics:
        # group by item
        grouped = df_label.groupby(["block_id", "response"])

        # collect ratings per item
        item_ratings = []
        for _, g in grouped:
            ratings = g.sort_values("assignment_id")[metric].to_list()
            if len(ratings) >= 2:  # need at least 2 raters
                item_ratings.append(ratings)

        if len(item_ratings) == 0:
            continue

        # convert to ragged matrix (pad with NaN)
        max_len = max(len(r) for r in item_ratings)
        data = np.full((max_len, len(item_ratings)), np.nan)

        for i, ratings in enumerate(item_ratings):
            data[:len(ratings), i] = ratings

        alpha = krippendorff.alpha(
            reliability_data=data,
            level_of_measurement="ordinal"
        )

        alpha_rows.append({
            "Label": label,
            "Metric": metric,
            "Krippendorff_alpha": round(alpha, 4)
        })

alpha_per_label_df = pd.DataFrame(alpha_rows)



alpha_table = (
    alpha_per_label_df
        .pivot(
            index="Label",
            columns="Metric",
            values="Krippendorff_alpha"
        )
        .reset_index()
)

# Optional: order columns to match your mean table
alpha_table = alpha_table[
    [
        "Label",
        "Progression",
        "Conversational Control",
        "Outcome Relevance",
        "Probing Effectiveness",
        "Conformity",
    ]
]

display(alpha_table)

